# Naija-Speech — Results Analysis & Live Demo

Validate the fine-tuned STT model two ways:

1. **Layer 1 — per-clip receipts** (no GPU needed): join the zero-shot and fine-tuned
   transcripts saved during evaluation and *see* the corrections — plus the honest
   counterpoint (clips fine-tuning made worse).
2. **Layer 2 — live A/B demo**: run the same audio through the base model and the
   base+adapter model side by side. Fast on GPU; workable-but-slow on a laptop CPU.

**Needs:** the unpacked results bundle at `outputs/stt_whisper_turbo_results/`
(adapter + hypotheses CSVs). Layer 1 needs nothing else. Layer 2 downloads the
~1.6 GB base model on first run and streams a few test clips from HF.

In [ ]:
import os, sys

# Run from repo root or notebooks/ — normalise to repo root.
if not os.path.exists("outputs") and os.path.exists("../outputs"):
    os.chdir("..")
sys.path.insert(0, "src")

RESULTS = "outputs/stt_whisper_turbo_results/outputs"
ADAPTER = f"{RESULTS}/whisper_turbo_hf/adapter"
assert os.path.exists(f"{RESULTS}/zeroshot/hypotheses.csv"), "results bundle not found!"
print("results bundle OK:", RESULTS)

## Layer 1 — join the per-clip transcripts

`hypotheses.csv` was written by both eval scripts: one row per test clip with the
human reference and the model's transcript. Joining on `idx` (same split, same
order, same filters) puts *before* and *after* side by side.

In [ ]:
import pandas as pd

b = pd.read_csv(f"{RESULTS}/zeroshot/hypotheses.csv").rename(columns={"hypothesis": "zeroshot"})
a = pd.read_csv(f"{RESULTS}/whisper_turbo_hf/results/hypotheses.csv")[["idx", "hypothesis"]] \
      .rename(columns={"hypothesis": "finetuned"})
j = b.merge(a, on="idx")
print(f"{len(j)} clips | {(j.zeroshot != j.finetuned).sum()} transcripts changed by fine-tuning")
j.head(3)

In [ ]:
# Per-clip WER (lowercased, otherwise raw) so we can rank improvements.
import jiwer

def wer1(ref, hyp):
    ref, hyp = str(ref).lower().strip(), str(hyp).lower().strip()
    if not ref:
        return 0.0
    try:
        return jiwer.wer(ref, hyp)
    except Exception:
        return float("nan")

j["wer_zs"] = [wer1(r, h) for r, h in zip(j.reference, j.zeroshot)]
j["wer_ft"] = [wer1(r, h) for r, h in zip(j.reference, j.finetuned)]
j["delta"]  = j.wer_zs - j.wer_ft   # positive = fine-tuning helped

print(f"mean per-clip WER: zero-shot {j.wer_zs.mean():.3f} -> fine-tuned {j.wer_ft.mean():.3f}")
print(f"clips improved: {(j.delta > 0).sum()} | unchanged: {(j.delta == 0).sum()} | worse: {(j.delta < 0).sum()}")

### The showcase: badly wrong → perfect

Clips the zero-shot model mangled (WER > 30%) that the fine-tuned model got
**exactly right** — thesis/defense examples.

In [ ]:
fixed = j[(j.wer_zs > 0.3) & (j.wer_ft == 0) & (j.reference.str.split().str.len().between(5, 18))]
for _, r in fixed.sort_values("wer_zs", ascending=False).head(8).iterrows():
    print(f"[{r.macro_accent} | {r.domain}]  (zero-shot WER {r.wer_zs:.0%})")
    print(f"  REF   : {r.reference}")
    print(f"  before: {r.zeroshot}")
    print(f"  after : {r.finetuned}\n")

### The honest counterpoint: where fine-tuning hurt

A few percent of clips getting worse is normal — worth knowing which ones
(and it strengthens §5.4's credibility to report it).

In [ ]:
worse = j[j.delta < -0.2].sort_values("delta")
print(f"clips notably worse (per-clip WER rose >20 pts): {len(worse)} of {len(j)} ({len(worse)/len(j):.1%})\n")
for _, r in worse.head(5).iterrows():
    print(f"[{r.macro_accent} | {r.domain}]  ({r.wer_zs:.0%} -> {r.wer_ft:.0%})")
    print(f"  REF   : {r.reference}")
    print(f"  before: {r.zeroshot}")
    print(f"  after : {r.finetuned}\n")

### The headline chart: before vs after, per accent

In [ ]:
import matplotlib.pyplot as plt

base = pd.read_csv(f"{RESULTS}/zeroshot/baseline_results.csv")
ft   = pd.read_csv(f"{RESULTS}/whisper_turbo_hf/results/finetuned_results.csv")
m = base.merge(ft, on=["group", "key"], suffixes=("_zs", "_ft"))
acc = m[m.group == "macro_accent"].sort_values("wer_zs")

fig, ax = plt.subplots(figsize=(8, 4.5))
x = range(len(acc))
ax.bar([i - 0.2 for i in x], acc.wer_zs * 100, 0.4, label="zero-shot")
ax.bar([i + 0.2 for i in x], acc.wer_ft * 100, 0.4, label="fine-tuned")
ax.set_xticks(list(x)); ax.set_xticklabels(acc.key)
ax.set_ylabel("WER (%)"); ax.set_title("WER by macro-accent: zero-shot vs fine-tuned")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/stt_whisper_turbo_results/wer_by_accent.png", dpi=150)
plt.show()
print("saved -> outputs/stt_whisper_turbo_results/wer_by_accent.png  (thesis Figure candidate)")

## Layer 2 — live A/B demo

Runs the same audio through **base** and **base + adapter**. On a GPU this is
seconds per clip; on a laptop CPU expect ~1–3 minutes per clip (the model is
809M parameters) — keep `N_CLIPS` small locally.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_CLIPS = 3 if DEVICE == "cuda" else 2
print(f"device: {DEVICE}" + ("  (CPU: each clip takes a minute or three — patience)" if DEVICE == "cpu" else ""))

In [ ]:
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(ADAPTER)
base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3-turbo")
tuned = PeftModel.from_pretrained(
    WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3-turbo"), ADAPTER)
print("models loaded")

In [ ]:
# Stream a few test clips from HF (audio decoded at 16 kHz; no full download).
from datasets import Audio, load_dataset

stream = (load_dataset("Johniblazee/naija-speech-afrispeech-ng", split="test", streaming=True)
          .cast_column("audio", Audio(sampling_rate=16000))
          .filter(lambda d: 0.5 <= d <= 30.0, input_columns="duration"))
clips = list(stream.shuffle(seed=7, buffer_size=200).take(N_CLIPS))
print(f"{len(clips)} clips streamed")

In [ ]:
def transcribe(model, audio_array):
    model.to(DEVICE).eval()
    dtype = next(model.parameters()).dtype
    model.generation_config.max_length = None
    feats = processor.feature_extractor(
        audio_array, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device=DEVICE, dtype=dtype)
    with torch.no_grad():
        out = model.generate(input_features=feats, language="english",
                             task="transcribe", max_new_tokens=225)
    return processor.batch_decode(out, skip_special_tokens=True)[0]

import IPython.display as ipd
for c in clips:
    print(f"=== [{c['macro_accent']} | {c['domain']}] ===")
    print(f"  REF   : {c['text_raw']}")
    print(f"  before: {transcribe(base,  c['audio']['array'])}")
    print(f"  after : {transcribe(tuned, c['audio']['array'])}\n")
    display(ipd.Audio(c["audio"]["array"], rate=16000))

### Your own voice

Record a short clip (any format librosa can read), set the path, run.

In [ ]:
AUDIO_PATH = None   # e.g. r"C:\Users\you\recording.wav"

if AUDIO_PATH:
    import librosa
    audio, _ = librosa.load(AUDIO_PATH, sr=16000, mono=True)
    print(f"  before: {transcribe(base,  audio)}")
    print(f"  after : {transcribe(tuned, audio)}")
else:
    print("set AUDIO_PATH to try your own recording")

## Layer 3 — other architectures, zero-shot, on the same clips (qualitative)

How do *other* existing models handle these exact clips, untrained on our corpus?

- **XLS-R (CTC):** the pretrained base has **no ASR head** and cannot transcribe;
  the comparable artifact is a community **English-CTC fine-tune** evaluated
  zero-shot on Nigerian speech (`jonatasgrosman/wav2vec2-xls-r-1b-english`, ~1B).
  Label it exactly that way anywhere it's reported.
- **Parakeet-TDT** (`nvidia/parakeet-tdt-0.6b-v3`): requires NVIDIA NeMo —
  **Colab/pod only** (NeMo does not realistically install on Windows).

This is a *qualitative* few-clip comparison. Citable WER numbers for these
architectures = the parked zero-shot sweep (scripts on a pod, ~1–2 GPU-hours).

In [ ]:
# --- XLS-R English-CTC, zero-shot (works anywhere; ~4GB download, CPU-slow) ---
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

XLSR_ID = "jonatasgrosman/wav2vec2-xls-r-1b-english"
xlsr_proc = Wav2Vec2Processor.from_pretrained(XLSR_ID)
xlsr = Wav2Vec2ForCTC.from_pretrained(XLSR_ID).to(DEVICE).eval()

def transcribe_xlsr(audio_array):
    import torch as _t
    inputs = xlsr_proc(audio_array, sampling_rate=16000, return_tensors="pt")
    with _t.no_grad():
        logits = xlsr(inputs.input_values.to(DEVICE)).logits
    ids = logits.argmax(dim=-1)
    return xlsr_proc.batch_decode(ids)[0].lower()

for c in clips:
    print(f"=== [{c['macro_accent']} | {c['domain']}] ===")
    print(f"  REF               : {c['text_raw']}")
    print(f"  XLS-R-Eng (0-shot): {transcribe_xlsr(c['audio']['array'])}")
    print(f"  Whisper fine-tuned: {transcribe(tuned, c['audio']['array'])}\n")

In [ ]:
# --- Parakeet-TDT zero-shot (COLAB/POD ONLY: needs NeMo; skipped gracefully) ---
try:
    import nemo.collections.asr as nemo_asr  # pip install -q "nemo_toolkit[asr]"
except ImportError:
    nemo_asr = None
    print("NeMo not installed - run on Colab/pod with:  pip install -q nemo_toolkit[asr]")

if nemo_asr:
    import soundfile as sf, tempfile, os as _os
    parakeet = nemo_asr.models.ASRModel.from_pretrained("nvidia/parakeet-tdt-0.6b-v3")
    tmpdir = tempfile.mkdtemp(prefix="parakeet_clips_")   # private dir, no name collisions
    paths = []
    for i, c in enumerate(clips):   # NeMo transcribes from files
        p = _os.path.join(tmpdir, f"clip{i}.wav")
        sf.write(p, c["audio"]["array"], 16000)
        paths.append(p)
    outs = parakeet.transcribe(paths)
    for c, o in zip(clips, outs):
        print(f"=== [{c['macro_accent']}] ===")
        print(f"  REF               : {c['text_raw']}")
        print(f"  Parakeet (0-shot) : {getattr(o, 'text', o)}")
        print(f"  Whisper fine-tuned: {transcribe(tuned, c['audio']['array'])}\n")